In [ ]:
import numpy as np
import pandas as pd

## MAP: Filtered and labels fixed

In [ ]:
rename_dict = {
    'Unclassified_Error': 'Unclassified Error',
    'Incomplete': 'Incomplete Calculation',
    'WNB': 'Whole Number Bias',
    'SwapDividend': 'Swapped Dividend',
    'Mult': 'Multiplication Error',
    'FlipChange': 'Flip and Change Error',
    'Irrelevant': 'Irrelevant Explanation',
    'Wrong_Fraction': 'Wrong Fraction',
    'Wrong_fraction': 'Wrong Fraction',  # merges the duplicate
    'Additive': 'Additive Reasoning Error',
    'Not_variable': 'Not A Variable',
    'Adding_terms': 'Adding Unlike Terms',
    'Inverse_operation': 'Inverse Operation Error',
    'Inversion': 'Inversion Error',
    'Duplication': 'Duplication Error',
    'Wrong_Operation': 'Wrong Operation',
    'Whole_numbers_larger': 'Whole Numbers Are Larger',
    'Longer_is_bigger': 'Longer Decimals Are Bigger', # Added 'Decimals' for semantic clarity
    'Ignores_zeroes': 'Ignores Zeroes',
    'Shorter_is_bigger': 'Shorter Decimals Are Bigger', # Added 'Decimals' for semantic clarity
    'Adding_across': 'Adding Fractions Across',
    'Denominator-only_change': 'Denominator Only Change',
    'Incorrect_equivalent_fraction_addition': 'Incorrect Equivalent Fraction Addition',
    'Division': 'Division Error',
    'Subtraction': 'Subtraction Error',
    'Unknowable': 'Unknowable Error',
    'Definition': 'Definition Error',
    'Interior': 'Interior Angle Error',
    'Positive': 'Positive/Negative Sign Error',
    'Tacking': 'Tacking On Zeroes', # may be misleading
    'Wrong_term': 'Wrong Term',
    'Firstterm': 'First Term Error',
    'Base_rate': 'Base Rate Error',
    'Multiplying_by_4': 'Multiplying By 4',
    'Certainty': 'Certainty Bias',
    'Scale': 'Scale Factor Error'
}

semantic_gemini_dict = {
    # The Unclassified / Vague Splits (Assuming you handle the splitting elsewhere,
    # but for now we label the generic bucket accurately)
    'Unclassified_Error': 'Vague Explanation or Unclassified Error',

    # Fraction & Calculation Errors
    'Incomplete': 'Incomplete Calculation or Failing to Simplify',
    'WNB': 'Treating Fraction Parts as Independent Whole Numbers',
    'Adding_across': 'Adding Numerators and Denominators Straight Across',
    'Denominator-only_change': 'Finding Common Denominator but Forgetting to Scale Numerators',
    'Incorrect_equivalent_fraction_addition': 'Adding Denominators After Finding the Common Denominator',
    'Duplication': 'Multiplying Both Numerator and Denominator by the Whole Number',
    'Inversion': 'Multiplying Whole Number by Denominator Instead of Numerator',
    'FlipChange': 'Inverting the Wrong Fraction in Division',
    'SwapDividend': 'Reversing the Order of Division to Divide by the Smaller Number',
    'Mult': 'Multiplying Instead of Dividing',
    'Wrong_Operation': 'Treating Fraction Multiplication as Mixed Number Addition',
    'Division': 'Dividing Instead of Multiplying to Find a Fraction Of an Amount',
    'Subtraction': 'Subtracting Instead of Multiplying to Find a Fraction Of an Amount',

    # Algebra & Equation Errors
    'Not_variable': 'Treating Variables as Place Value Digits',
    'Adding_terms': 'Treating Algebraic Coefficients as Addition',
    'Inverse_operation': 'Applying the Wrong Inverse Operation',

    # Reading Comprehension & Irrelevant Facts
    'Wrong_Fraction': 'Solving for the Given Item Instead of the Requested Item',
    'Wrong_fraction': 'Solving for the Given Item Instead of the Requested Item',
    'Irrelevant': 'Using Irrelevant Mathematical Facts or Surface Features',

    # Decimals & Positives/Negatives
    'Whole_numbers_larger': 'Believing Whole Numbers are Automatically Larger than Decimals',
    'Longer_is_bigger': 'Believing Longer Decimals are Larger',
    'Shorter_is_bigger': 'Believing Shorter Decimals are Larger',
    'Ignores_zeroes': 'Ignoring Decimal Placeholder Zeroes',
    'Positive': 'Overgeneralizing Two Negatives Make a Positive to Addition and Subtraction',
    'Tacking': 'Tacking On Signs After Absolute Value Calculation',

    # Sequences & Proportions
    'Wrong_term': 'Calculating the Next Term Instead of the Requested Nth Term',
    'Firstterm': 'Multiplying Term Number by First Term Instead of Using Sequence Rule',
    'Additive': 'Using Additive Reasoning instead of Multiplicative Proportions',
    'Base_rate': 'Treating Inverse Proportion as Direct Proportion by Dividing',
    'Multiplying_by_4': 'Applying Direct Scaling to an Inverse Proportion Problem',

    # Geometry & Probability
    'Unknowable': 'Believing Regular Polygon Properties Cannot Be Found From One Angle',
    'Definition': 'Misunderstanding the Definition of a Regular Polygon',
    'Interior': 'Confusing Interior Angle Sum Formulas',
    'Certainty': 'Confusing High Probability with Absolute Certainty',
    'Scale': 'Misunderstanding the 0 to 1 Probability Scale',
}

semantic_chatgpt_dict = {
    # --- Meta / weak signals ---
    'Unclassified_Error': 'Unclear Reasoning',
    'Incomplete': 'Incomplete Reasoning',
    'Unknowable': 'Insufficient Information Provided',
    'Irrelevant': 'Irrelevant Explanation',

    # --- Core conceptual misconceptions ---
    'WNB': 'Whole Number Bias',
    'Whole_numbers_larger': 'Whole Number Bias',
    'Longer_is_bigger': 'Decimal Length Bias',
    'Shorter_is_bigger': 'Decimal Length Bias',
    'Additive': 'Additive Instead of Multiplicative Reasoning',
    'Base_rate': 'Base Rate Neglect',
    'Certainty': 'Overconfidence Bias',

    # --- Fractions ---
    'Adding_across': 'Incorrect Fraction Addition (Across Numerator and Denominator)',
    'Wrong_Fraction': 'Incorrect Fraction Representation',
    'Wrong_fraction': 'Incorrect Fraction Representation',
    'Incorrect_equivalent_fraction_addition': 'Incorrect Equivalent Fraction Reasoning',
    'Denominator-only_change': 'Incorrect Fraction Scaling',

    # --- Operations ---
    'Wrong_Operation': 'Incorrect Operation Selection',
    'Inverse_operation': 'Incorrect Use of Inverse Operation',
    'Division': 'Division Procedure Error',
    'Multiplication': 'Multiplication Procedure Error',
    'Subtraction': 'Subtraction Procedure Error',
    'Mult': 'Multiplication Procedure Error',

    # --- Algebra ---
    'Not_variable': 'Misunderstanding of Variables',
    'Adding_terms': 'Combining Unlike Terms',
    'Wrong_term': 'Incorrect Algebraic Term',
    'Firstterm': 'Misidentification of Term Structure',

    # --- Sign / negatives ---
    'Positive': 'Sign Misinterpretation (Positive/Negative)',
    'Inversion': 'Incorrect Sign Inversion',

    # --- Procedural quirks (reframed!) ---
    'SwapDividend': 'Incorrect Operand Order in Division',
    'FlipChange': 'Incorrect Fraction Inversion Procedure',
    'Duplication': 'Unnecessary Duplication of Terms',
    'Tacking': 'Appending Digits Without Justification',

    # --- Geometry ---
    'Interior': 'Misunderstanding of Interior Angles',
    'Scale': 'Scale Factor Misunderstanding',

    # --- Misc ---
    'Definition': 'Incorrect Definition Recall',
    'Ignores_zeroes': 'Place Value Misunderstanding',
    'Multiplying_by_4': 'Incorrect Constant Multiplication Pattern'
}

In [ ]:
raw_train_df = pd.read_csv("/content/map_train.csv")
df = raw_train_df[~raw_train_df['Category'].isin(['True_Correct', 'False_Correct'])].copy()

# Fill NaNs
df['Misconception'] = df['Misconception'].fillna('Unclassified_Error').astype(str)

# Map the classes to readable English
df['Misconception'] = df['Misconception'].replace(rename_dict)

# Fix the hidden newline CSV bug
text_columns = ['QuestionText', 'MC_Answer', 'StudentExplanation']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(r'[\r\n]+', ' ', regex=True)

df.to_csv("/content/map_ready.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
df['Misconception'].unique()

array(['Unclassified Error', 'Incomplete Calculation',
       'Whole Number Bias', 'Swapped Dividend', 'Multiplication Error',
       'Flip and Change Error', 'Irrelevant Explanation',
       'Wrong Fraction', 'Additive Reasoning Error', 'Not A Variable',
       'Adding Unlike Terms', 'Inverse Operation Error',
       'Inversion Error', 'Duplication Error', 'Wrong Operation',
       'Whole Numbers Are Larger', 'Longer Decimals Are Bigger',
       'Ignores Zeroes', 'Shorter Decimals Are Bigger',
       'Adding Fractions Across', 'Denominator Only Change',
       'Incorrect Equivalent Fraction Addition', 'Division Error',
       'Subtraction Error', 'Unknowable Error', 'Definition Error',
       'Interior Angle Error', 'Positive/Negative Sign Error',
       'Tacking On Zeroes', 'Wrong Term', 'First Term Error',
       'Base Rate Error', 'Multiplying By 4', 'Certainty Bias',
       'Scale Factor Error'], dtype=object)

## MAP: Data split for validation

In [ ]:
import csv # for strict quoting
from sklearn.model_selection import train_test_split

# Load Data
df = pd.read_csv("/content/map_ready.csv")


# Filter out ultra-rare classes (< 10 samples)
class_counts = df['Misconception'].value_counts()
valid_classes = class_counts[class_counts >= 10].index.tolist()
df = df[df['Misconception'].isin(valid_classes)].copy()

print(f"Total rows after filtering: {len(df)}")
print(f"Total unique classes remaining: {len(valid_classes)}")

# Stratified Train/Validation Split (95/5)
train_df, val_df = train_test_split(
    df,
    test_size=0.05,
    random_state=99,
    stratify=df['Misconception']
)

# Lock them into CSV files with STRICT QUOTING
# Needed to prevent symbols from braking csv cells
train_df.to_csv("/content/train_BLvsLLM.csv", index=False, quoting=csv.QUOTE_ALL)
val_df.to_csv("/content/val_BLvsLLM.csv", index=False, quoting=csv.QUOTE_ALL)

print("\nData successfully locked and safely saved!")
print(f"Train set size: {len(train_df)} rows")
print(f"Validation set size: {len(val_df)} rows")

Total rows after filtering: 21652
Total unique classes remaining: 33

Data successfully locked and safely saved!
Train set size: 20569 rows
Validation set size: 1083 rows
